# P02 Ошибка и выбор параметров

**Эталон решения**. Полное объяснение находится в Pages. Этот файл содержит исполняемый практикум.

## Цель

Посчитать MSE, сравнить настройки и отделить алгоритм подбора от выбранной модели.

**Перед началом:** P01: модель и параметры; среднее арифметическое и квадрат числа.

## Подготовка

Запускайте из корня репозитория в его виртуальном окружении. Первичная установка зависимостей описана в README. После установки выполнение не использует сеть, платные API или GPU.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "mlcourse" / "pilot.py").is_file():
    if ROOT == ROOT.parent:
        raise RuntimeError("Откройте Jupyter из корня ml-llm-course")
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from mlcourse.pilot import (
    mse, predict_line, linear_gradient, central_difference, fit_standardizer,
    matvec, fit_gradient, make_dataset, run_experiment, predict_payload,
)
plt.rcParams.update({"figure.figsize": (7.2, 4.2), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
FIGURES = ROOT / "artifacts" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("Данные: искусственные учебные примеры. CPU. Состояние ученика не оценивается автоматически.")

## Разобранный пример

MSE — среднее квадратов разностей предсказания и известного ответа. Ошибки разных знаков не сокращаются. Выбирая параметры по ошибке на примерах, мы выполняем простейшее обучение; при этом выбранная метрика выражает лишь часть требований задачи.

### Разобрать вычисление ошибки

In [ ]:
x = np.array([1., 2., 3.]); y = np.array([3., 5., 7.])
pred_a, pred_b = predict_line(x, 2, 1), predict_line(x, 1, 1)
display(pd.DataFrame({"y": y, "B": pred_b, "остаток B-y": pred_b-y, "квадрат": (pred_b-y)**2}))
print("MSE A:", mse(y, pred_a), "MSE B:", mse(y, pred_b))
assert np.isclose(mse(y, pred_b), 14/3)

### Подобрать параметры среди кандидатов

In [ ]:
candidates = [(1, 1), (2, 0), (2, 1)]
scores = [(w, b, mse(y, predict_line(x, w, b))) for w, b in candidates]
display(pd.DataFrame(scores, columns=["w", "b", "MSE train"]))
chosen = min(scores, key=lambda row: row[2])
assert chosen[:2] == (2, 1)

## Практика с поддержкой

Для параметров (2,0) посчитайте каждую ошибку вручную. Должно получиться три одинаковых квадрата; объясните причину.

## Самостоятельная задача

Реализуйте собственную функцию mse(actual, predicted), включая проверки пустых списков, одинаковой длины и конечности чисел. Проверьте её на новых y=[5,8,11] с предсказаниями двух кандидатов из P01. Для CLI сохраните функцию в отдельный .py файл и выполните tools/check_submission.py <файл> --task mse.

Сначала выполните свою попытку. Подсказки и разбор доступны у наставника; использованную помощь отмечайте.

### Эталон

Открывайте после собственной попытки. После разбора понадобится новый вариант для проверки самостоятельности.

In [ ]:
def mse_answer(actual, predicted):
    if len(actual) == 0 or len(actual) != len(predicted):
        raise ValueError("Нужны непустые списки одинаковой длины")
    if not all(np.isfinite(v) for v in [*actual, *predicted]):
        raise ValueError("Нужны конечные числа")
    return sum((a-p)**2 for a, p in zip(actual, predicted)) / len(actual)
assert np.isclose(mse_answer([5, 8, 11], [4, 6, 8]), 14/3)
assert mse_answer([5, 8, 11], [5, 8, 11]) == 0
print("Новый набор: MSE неверного кандидата 14/3, верного 0.")

## Проверка

Формула использует квадрат, а не знак или модуль; знаменатель — число примеров; некорректные входы отклоняются; минимум ошибки интерпретируется в рамках данных.

Успешный Run All подтверждает запуск подготовленных примеров. Самостоятельная работа и понимание требуют отдельной проверки наставником.

## Следующий шаг

Посчитайте MSE для actual=[0,2], predicted=[1,1]. Затем объясните отличие MSE=1 от средней ошибки=0.

[Основной источник](https://developers.google.com/machine-learning/crash-course/linear-regression/loss). Все значения в примерах синтетические. Сохраняйте исходные ноутбуки; для своей работы используйте копию.